# 🐜 ANT — GPU Training

**937K parameter byte-level transformer** where ALL knowledge lives in persistent trie memory.

## 3-Phase Curriculum
1. **Phase A** — Base LM (no memory, all weights trainable)
2. **Phase B** — Memory Training (frozen base, train AddrNet/V_proj/tags)
3. **Phase C** — End-to-End (trie READ+WRITE every forward pass)

## Architecture
- `model.py` — Pure NN (937K params, stores NO knowledge)
- `engine.py` — Orchestrates per-token READ→PROCESS→WRITE trie cycles
- `ant_memory/` — Rust trie (PyO3, arena-allocated, 256-ary, 8 levels)
- `train.py` — Phase A/B/C curriculum using engine

## 1. Setup — GPU, Rust, Dependencies

In [ ]:
# Verify GPU
!nvidia-smi

# Install Python dependencies
!pip install -q datasets numpy torch huggingface_hub maturin

# Install Rust toolchain (needed for ant_memory)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Clone repo, build Rust extension, HuggingFace login
import os

REPO_DIR = '/content/ANT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaaninel/ANT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# Build ant_memory Rust extension
os.environ['PYO3_USE_ABI3_FORWARD_COMPATIBILITY'] = '1'
!cd ant_memory && maturin develop --release

# Verify build
import ant_memory
print('\u2713 ant_memory Rust extension built successfully')

# HuggingFace login (set HF_TOKEN in Colab secrets)
from huggingface_hub import login, create_repo
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

HF_REPO = 'kaaninel/ANT'
try:
    create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'\u2713 HF repo: https://huggingface.co/{HF_REPO}')
except: pass

## 2. Configuration

In [ ]:
# ================================================================
# TRAINING CONFIG — edit and re-run to adjust
# ================================================================

# Steps per phase
STEPS_A = 5000    # Phase A: Base LM only (no memory)
STEPS_B = 3000    # Phase B: Frozen base, train AddrNet/V_proj/tags
STEPS_C = 5000    # Phase C: End-to-end with trie READ+WRITE

# Learning rates
LR_A = 3e-4
LR_B = 1e-3       # Higher LR for memory subsystem only
LR_C = 1e-4       # Lower LR for end-to-end fine-tuning

# Batch sizes (GPU allows larger)
BATCH_A = 32
BATCH_B = 32
BATCH_C = 16

CKPT_DIR = '/content/ant_checkpoints/train'

# A100/GPU optimizations
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

total = STEPS_A + STEPS_B + STEPS_C
print(f'Total steps: {total:,}')
print(f'Phase A: {STEPS_A} steps, batch={BATCH_A}, lr={LR_A}')
print(f'Phase B: {STEPS_B} steps, batch={BATCH_B}, lr={LR_B}')
print(f'Phase C: {STEPS_C} steps, batch={BATCH_C}, lr={LR_C}')

## 3. Train

All 3 phases run automatically using `ANTEngine` + `phase_a/b/c`.
Checkpoints saved every 500 steps.

In [ ]:
import os
import torch

from config import ModelConfig, MemoryConfig
from model import ANT
from engine import ANTEngine
from train import phase_a, phase_b, phase_c, count_params

device = 'cuda'

# Build model + engine
cfg = ModelConfig()
mem_cfg = MemoryConfig()
mem_cfg.data_path = os.path.join(CKPT_DIR, 'memory')
os.makedirs(CKPT_DIR, exist_ok=True)

model = ANT(cfg).to(device)
engine = ANTEngine(model, mem_cfg, device=device)

total_params = count_params(model, trainable_only=False)
print(f'ANT model: {total_params:,} parameters')
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')

# Phase A — Pure LM
phase_a(engine, cfg, device,
        steps=STEPS_A, lr=LR_A, batch_size=BATCH_A,
        ckpt_dir=CKPT_DIR)

# Phase B — Memory Training
phase_b(engine, cfg, device,
        steps=STEPS_B, lr=LR_B, batch_size=BATCH_B,
        ckpt_dir=CKPT_DIR)

# Phase C — End-to-End
phase_c(engine, cfg, device,
        steps=STEPS_C, lr=LR_C, batch_size=BATCH_C,
        ckpt_dir=CKPT_DIR)

engine.flush()
print('\nTraining complete!')

## 4. Test Generation

Quick test of text generation with trie memory.

In [ ]:
import torch
import os
from config import ModelConfig, MemoryConfig
from model import ANT
from engine import ANTEngine
from train import load_checkpoint
from data import tokenize, detokenize, BOS_ID, EOS_ID, generate_dataset, ANS_ID

# Load best checkpoint
cfg = ModelConfig()
mem_cfg = MemoryConfig()
mem_cfg.data_path = os.path.join(CKPT_DIR, 'memory')

model = ANT(cfg).cuda()
engine = ANTEngine(model, mem_cfg, device='cuda')

best_ckpt = os.path.join(CKPT_DIR, 'checkpoint_phaseC.pt')
if not os.path.exists(best_ckpt):
    best_ckpt = os.path.join(CKPT_DIR, 'checkpoint_latest.pt')
step, phase = load_checkpoint(model, None, best_ckpt, 'cuda')
print(f'Loaded phase {phase}, step {step}')
print(f'Trie: {engine.memory_stats()}')

# --- Text generation ---
print('\n=== Text Generation ===')
for p in ['The ', 'In the year ', '$ ls -la ']:
    prompt_ids = [BOS_ID] + tokenize(p)
    generated = engine.generate(prompt_ids, max_tokens=100,
                                temperature=0.7, top_k=30)
    text = detokenize(generated)
    print(f'\n--- {repr(p)} ---')
    print(text[:200])

# --- Memory QA ---
print('\n=== Memory QA ===')
examples = generate_dataset(5, seed=99, tagged=False)
for ex in examples:
    # Write passage to trie
    p_ids = [BOS_ID] + tokenize(ex.passage)[:cfg.max_seq_len - 2] + [EOS_ID]
    p_tensor = torch.tensor([p_ids], dtype=torch.long, device='cuda')
    engine.reset_state(1)
    with torch.no_grad():
        engine.encode(p_tensor, write_to_trie=True)

    # Generate answer
    q_ids = [BOS_ID] + tokenize(ex.question + chr(ANS_ID))
    generated = engine.generate(q_ids, max_tokens=50,
                                temperature=0.3, top_k=10)
    answer = detokenize(generated)
    print(f'Q: {ex.question}')
    print(f'A: {answer[:100]}  (expected: {ex.answer})')
    print()

## 5. Upload Final Checkpoint

In [ ]:
from huggingface_hub import HfApi
import os
api = HfApi()

for f in ['checkpoint_latest.pt', 'checkpoint_phaseA.pt',
          'checkpoint_phaseB.pt', 'checkpoint_phaseC.pt']:
    path = os.path.join(CKPT_DIR, f)
    if os.path.exists(path):
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f'checkpoints/train/{f}',
            repo_id='kaaninel/ANT', repo_type='model')
        print(f'Uploaded {f}')

print('\nDone! Check https://huggingface.co/kaaninel/ANT')